# OdontoCA clinical submission reproduction

Secondary analysis of the public Single-tooth ECC cohort. Research only.
This notebook reproduces the manuscript analysis: 996 transitions, 83 events, 81 children, child-grouped nested evaluation and training-only recalibration. Image, microbiome, synthetic and FHIR branches are excluded from this submission workflow.

Source data are downloaded from the source authors, checked against a fixed SHA-256 and kept locally. This package distributes aggregate outputs; it does not grant rights to redistribute the original participant-level worksheet.

Run all cells in order. The last cell verifies the exported results. Author, ethics and funding declarations still require author confirmation before submission.


In [ ]:
# Fixed manuscript analysis settings. Change only the optional source path.
PROFILE_UI = "CLINICAL_INTERNAL"
SEED_UI = 20260930
OUTER_FOLDS_UI = 5
INNER_FOLDS_UI = 3
BOOTSTRAPS_UI = 1000
STRICT_REPRODUCIBILITY_UI = True
RUN_PRIMARY_ANALYSIS_UI = True
RUN_LEGACY_SPLIT_UI = False
LOCAL_TABLE_S1_UI = "" #@param {type:"string"}


In [ ]:
# 3. Configuração central
from dataclasses import dataclass, asdict
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, random, hashlib, platform, subprocess, re, zipfile, warnings, shutil

@dataclass
class Config:
    PROFILE: str = str(globals().get("PROFILE_UI", "CI_SMOKE"))
    SEED: int = int(globals().get("SEED_UI", 20260906))
    STRICT: bool = bool(globals().get("STRICT_REPRODUCIBILITY_UI", True))
    DOWNLOAD_IMAGE_BENCHMARK: bool = False
    TRAIN_IMAGE_MODEL: bool = False
    IMAGE_EPOCHS: int = 30
    IMAGE_SIZE: int = 640
    SEGMENT_ANY_TOOTH: bool = False
    SEGMENT_WEIGHT_DIR: str = "/content/segmentanytooth_weights"
    QIITA_STUDY_ID: int = 14341
    QIITA_PREFERRED_ARTIFACT_ID: str = "214894"
    LOCAL_TABLE_S1: str = str(globals().get("LOCAL_TABLE_S1_UI", ""))
    LOCAL_QIITA_ZIP: str = ""

CFG = Config()

if CFG.PROFILE == "CI_SMOKE":
    OUTER_FOLDS = 3
    INNER_FOLDS = 2
    BOOTSTRAPS = min(int(globals().get("BOOTSTRAPS_UI", 200)), 200)
else:
    OUTER_FOLDS = int(globals().get("OUTER_FOLDS_UI", 5))
    INNER_FOLDS = int(globals().get("INNER_FOLDS_UI", 3))
    BOOTSTRAPS = int(globals().get("BOOTSTRAPS_UI", 1000 if CFG.PROFILE == "CLINICAL_INTERNAL" else 5000))

ALLOW_NETWORK = CFG.PROFILE != "CI_SMOKE"
LOAD_REAL_CLINICAL = CFG.PROFILE in {"CLINICAL_INTERNAL","REPRO_DATASET","MICROBIOME_RESEARCH","FULL_RESEARCH"}
LOAD_REAL_MICRO = CFG.PROFILE in {"MICROBIOME_RESEARCH","FULL_RESEARCH"}
AUDIT_IMAGE_DATASET = CFG.PROFILE in {"IMAGE_DATASET_AUDIT","FULL_RESEARCH"}

IN_COLAB = "google.colab" in sys.modules
BASE = Path("/content/OdontoCA_v1_3_5" if IN_COLAB else "./OdontoCA_outputs").resolve()
DIRS = {
    "raw":BASE/"00_raw",
    "clinical":BASE/"01_clinical",
    "micro":BASE/"02_microbiome",
    "image":BASE/"03_image",
    "models":BASE/"04_models",
    "figures":BASE/"05_figures",
    "reports":BASE/"06_reports",
    "tests":BASE/"07_tests",
    "exports":BASE/"08_exports",
}
for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

random.seed(CFG.SEED)
os.environ["PYTHONHASHSEED"] = str(CFG.SEED)

SOURCE_NOTEBOOK_HASHES = {"OdontoCA_v1_2_SDD_TDD_MULTIMODAL_VALIDATED_EXECUTED(1).ipynb": "0bbaa310191e3366b5ffe61c261b23219b1e84c73d48dd4c65711be73255eb77", "SingleTooth_ECC_FULL_PIPELINE_v3_2_AUDIT_PATCH(1).ipynb": "ea7ce5f358b536a821d930e7a07dee454914e63a21ac132139b0725a6f0e2b84"}

TARGETS = {
    "PR_AUC_MIN":0.78,
    "ROC_AUC_MIN":0.88,
    "BRIER_MAX":0.085,
    "DICE_MIN":0.82,
}

print(json.dumps({
    "profile":CFG.PROFILE,
    "outer_folds":OUTER_FOLDS,
    "inner_folds":INNER_FOLDS,
    "bootstraps":BOOTSTRAPS,
    "base":str(BASE),
    "primary_analysis": bool(globals().get("RUN_PRIMARY_ANALYSIS_UI", True)),
    "legacy_split": bool(globals().get("RUN_LEGACY_SPLIT_UI", False)),
}, indent=2))


In [ ]:
# 4. Dependências — lazy install para não penalizar CI_SMOKE
import importlib.util

BASE_PACKAGES = {
    "numpy":"numpy",
    "pandas":"pandas",
    "scikit-learn":"sklearn",
    "openpyxl":"openpyxl",
    "requests":"requests",
    "matplotlib":"matplotlib",
    "scipy":"scipy",
}

def ensure_packages(mapping):
    missing = [pkg for pkg,mod in mapping.items() if importlib.util.find_spec(mod) is None]
    if missing:
        if not ALLOW_NETWORK:
            raise RuntimeError("Pacotes ausentes no CI_SMOKE: " + ", ".join(missing))
        subprocess.run([sys.executable,"-m","pip","install","-q",*missing], check=True)
    return missing

# Pin the tested dependency versions before importing scientific packages.
import importlib.metadata
PINNED_VERSIONS = {'numpy': '2.3.5', 'pandas': '2.2.3', 'scipy': '1.17.0', 'scikit-learn': '1.8.0', 'matplotlib': '3.10.8', 'openpyxl': '3.1.5', 'requests': '2.34.2', 'pillow': '12.3.0'}
missing_or_different = []
for package, expected in PINNED_VERSIONS.items():
    try:
        observed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        observed = None
    if observed != expected:
        missing_or_different.append(f"{package}=={expected}")
if missing_or_different:
    subprocess.run([sys.executable,"-m","pip","install","-q",*missing_or_different],check=True)
ensure_packages(BASE_PACKAGES)

import numpy as np
import pandas as pd
import scipy
import matplotlib.pyplot as plt
import sklearn

from scipy.stats import beta as beta_dist
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss, log_loss,
    precision_recall_curve, roc_curve
)

warnings.filterwarnings("ignore")

In [ ]:
MASTER_SCHEMA = {
    "patient_id":{"required":True},
    "visit_id":{"required":True},
    "visit_timestamp":{"required":True},
    "tooth_fdi":{"required":True},
    "surface_code":{"required":False,"domain":{"O","V","L","P","M","D"}},
    "image_path":{"required":False},
    "mask_path":{"required":False},
    "visual_lesion_class":{"required":False,"domain":{0,1,2,3}},
    "lesion_area_px":{"required":False,"min":0.0},
    "plaque_coverage_pct":{"required":False,"min":0.0,"max":100.0},
    "next_visit_class":{"required":False,"domain":{0,1,2}},
}

VALIDATED_TARGETS = {"caries"}
EXPERIMENTAL_TARGETS = {"plaque","surface_diagnosis"}

class UnsupportedGroundTruth(RuntimeError):
    pass

class CrossCohortFusionError(RuntimeError):
    pass

def require_validated_target(target):
    t = str(target).strip().lower()
    if t not in VALIDATED_TARGETS:
        raise UnsupportedGroundTruth(f"Target não validado: {t}")
    return t

def validate_master_schema(df):
    errors = []
    for k,spec in MASTER_SCHEMA.items():
        if spec.get("required") and k not in df.columns:
            errors.append(f"missing:{k}")
    if errors:
        return {"ok":False,"errors":errors}
    if df.duplicated(["patient_id","visit_id","tooth_fdi"]).any():
        errors.append("duplicate_patient_visit_tooth")
    if "surface_code" in df:
        bad = set(df["surface_code"].dropna().astype(str)) - MASTER_SCHEMA["surface_code"]["domain"]
        if bad: errors.append(f"bad_surface:{sorted(bad)}")
    if "lesion_area_px" in df:
        x = pd.to_numeric(df["lesion_area_px"],errors="coerce")
        if (x<0).fillna(False).any(): errors.append("negative_lesion_area")
    if "plaque_coverage_pct" in df:
        x = pd.to_numeric(df["plaque_coverage_pct"],errors="coerce")
        if ((x<0)|(x>100)).fillna(False).any(): errors.append("plaque_range")
    return {"ok":not errors,"errors":errors}

UPPER = ["T55","T54","T53","T52","T51","T61","T62","T63","T64","T65"]
LOWER = ["T85","T84","T83","T82","T81","T71","T72","T73","T74","T75"]
ARCHES = {"upper":UPPER,"lower":LOWER}
ALL_PRIMARY = UPPER + LOWER

NEIGHBORS = {}
for seq in ARCHES.values():
    for i,t in enumerate(seq):
        NEIGHBORS[t] = (
            ([seq[i-1]] if i>0 else []) +
            ([seq[i+1]] if i<len(seq)-1 else [])
        )

ANTAGONISTS = {
    "T55":"T85","T54":"T84","T53":"T83","T52":"T82","T51":"T81",
    "T61":"T71","T62":"T72","T63":"T73","T64":"T74","T65":"T75",
}
ANTAGONISTS.update({v:k for k,v in list(ANTAGONISTS.items())})

VIEW_ALIASES = {
    "front":"front","frontal":"front",
    "left":"left","left lateral":"left","lateral esquerda":"left",
    "right":"right","right lateral":"right","lateral direita":"right",
    "upper":"upper","maxillary occlusal":"upper","oclusal superior":"upper",
    "lower":"lower","mandibular occlusal":"lower","oclusal inferior":"lower",
}

def normalize_view(view):
    k = re.sub(r"\s+"," ",str(view).strip().lower())
    if k not in VIEW_ALIASES:
        raise ValueError(f"Vista inválida: {view}")
    return VIEW_ALIASES[k]

def build_odontogram(records):
    d = pd.DataFrame(records)
    req = {"patient_id","visit_id","tooth_fdi","state"}
    assert not (req-set(d.columns)), f"Campos ausentes: {sorted(req-set(d.columns))}"
    d["tooth_fdi"] = d["tooth_fdi"].astype(str).str.replace("T","",regex=False)
    valid = {x[1:] for x in ALL_PRIMARY}
    bad = set(d["tooth_fdi"]) - valid
    assert not bad, f"FDI inválido: {sorted(bad)}"
    assert not d.duplicated(["patient_id","visit_id","tooth_fdi"]).any()
    return d.sort_values(["patient_id","visit_id","tooth_fdi"]).reset_index(drop=True)

In [ ]:
SOURCE_COMMIT = "e5868fe5664460c7aac1c5b6d7980776ad26b29c"
SOURCE_SHA256 = "b7819fee81efbe2e227b7700c3cd86ce8bc6f7fe6f49da6214f4107d099184fa"
SUPPLEMENT_URLS = {
    "Table_S1.xlsx": (
        "https://raw.githubusercontent.com/HuangShiLab/Single-tooth-ECC/"
        + SOURCE_COMMIT + "/Figures_and_Tables_in_Manuscript/Table_S1.xlsx"
    ),
}
REQUIRED_SOURCE = [
    "SampleID","HostGroup","Status_Host","Status_Tooth",
    "dmfs_Host","dmfs_Tooth","Month","Timepoint","Position",
    "NicheICM","NicheUL","NicheLR","HostID",
    "Future_Status_Tooth","NicheAP","Position_Tooth",
    "X","Y","Z","spatial_dist_weighted_mean_dmfs",
    "sum_dmfs","sum_ns_dt","sum_s_dt","Time_to_decay"
]
FORBIDDEN_PREDICTORS = {"HostGroup","Future_Status_Tooth","Time_to_decay",
                        "future_status_source","time_to_decay_source",
                        "state_t1","progression_event"}

def sha256_file(path,chunk=1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(chunk),b""):
            h.update(b)
    return h.hexdigest()

def network_guard():
    if not ALLOW_NETWORK:
        raise RuntimeError("Rede bloqueada no CI_SMOKE.")

def requests_session():
    network_guard()
    import requests
    from requests.adapters import HTTPAdapter
    from urllib3.util.retry import Retry
    s=requests.Session()
    s.mount("https://",HTTPAdapter(max_retries=Retry(
        total=4,backoff_factor=1,status_forcelist=[429,500,502,503,504]
    )))
    return s

def stream_download(url,dest,timeout=180):
    s=requests_session()
    dest=Path(dest)
    tmp=dest.with_suffix(dest.suffix+".part")
    with s.get(url,stream=True,timeout=timeout) as r:
        r.raise_for_status()
        with open(tmp,"wb") as f:
            for chunk in r.iter_content(1024*1024):
                if chunk:f.write(chunk)
    tmp.replace(dest)
    return dest

def load_table_s1():
    target = DIRS["raw"]/"Table_S1.xlsx"
    local = Path(CFG.LOCAL_TABLE_S1) if CFG.LOCAL_TABLE_S1 else None
    if local and local.exists():
        shutil.copy2(local,target)
    if not target.exists():
        stream_download(SUPPLEMENT_URLS["Table_S1.xlsx"],target)
    observed_sha = sha256_file(target)
    if observed_sha != SOURCE_SHA256:
        raise RuntimeError(
            f"SHA-256 divergente para Table_S1.xlsx: {observed_sha}. "
            f"Esperado: {SOURCE_SHA256}"
        )
    xls=pd.ExcelFile(target)
    assert "all_metadata" in xls.sheet_names
    meta=pd.read_excel(target,sheet_name="all_metadata")
    meta.columns=[str(c).strip() for c in meta.columns]
    missing=[c for c in REQUIRED_SOURCE if c not in meta.columns]
    assert not missing,f"Campos ausentes: {missing}"
    return meta,target

In [ ]:
# 8. Mapa dentário e reconstrução longitudinal exata
def build_tooth_map(meta):
    rows=[]
    for arch,seq in ARCHES.items():
        for idx,tooth in enumerate(seq):
            src=meta.loc[meta["Position"].eq(tooth)]
            assert len(src)>0,tooth
            r=src.iloc[0]
            rows.append({
                "tooth_position":tooth,
                "tooth_fdi":int(tooth[1:]),
                "tooth_type_index":int(str(r["Position_Tooth"])[1:]),
                "arch":arch,"arch_index":idx,
                "niche_icm":r["NicheICM"],"niche_ul":r["NicheUL"],
                "niche_lr":r["NicheLR"],"niche_ap":r["NicheAP"],
                "x":float(r["X"]),"y":float(r["Y"]),"z":float(r["Z"]),
                "neighbor_prev_tooth":seq[idx-1] if idx>0 else np.nan,
                "neighbor_next_tooth":seq[idx+1] if idx<len(seq)-1 else np.nan,
            })
    return pd.DataFrame(rows)

def clean_single_tooth(meta):
    tooth_map=build_tooth_map(meta)
    composite=meta.loc[meta["Position"].eq("T5161")].copy()
    single=meta.loc[meta["Position"].isin(tooth_map["tooth_position"])].copy()
    single=single.merge(
        tooth_map,left_on="Position",right_on="tooth_position",
        how="left",validate="many_to_one"
    )
    clean=pd.DataFrame({
        "sample_id":single["SampleID"].astype(str),
        "child_id":single["HostID"].astype(str),
        "timepoint":single["Timepoint"].astype(str),
        "time_index":single["Timepoint"].astype(str).str.extract(r"(\d+)")[0].astype(int),
        "age_months":pd.to_numeric(single["Month"],errors="coerce"),
        "tooth_position":single["tooth_position"],
        "tooth_fdi":single["tooth_fdi"],
        "tooth_type_index":single["tooth_type_index"],
        "arch":single["arch"],"arch_index":single["arch_index"],
        "niche_icm":single["NicheICM"],"niche_ul":single["NicheUL"],
        "niche_lr":single["NicheLR"],"niche_ap":single["NicheAP"],
        "x":pd.to_numeric(single["X"],errors="coerce"),
        "y":pd.to_numeric(single["Y"],errors="coerce"),
        "z":pd.to_numeric(single["Z"],errors="coerce"),
        "tooth_status":single["Status_Tooth"].astype(str),
        "tooth_state_code":single["Status_Tooth"].map({"H":0,"C":1}),
        "tooth_dmfs":pd.to_numeric(single["dmfs_Tooth"],errors="coerce"),
        "host_status":single["Status_Host"].astype(str),
        "host_dmfs":pd.to_numeric(single["dmfs_Host"],errors="coerce"),
        "spatial_weighted_dmfs":pd.to_numeric(single["spatial_dist_weighted_mean_dmfs"],errors="coerce"),
        "sum_dmfs":pd.to_numeric(single["sum_dmfs"],errors="coerce"),
        "sum_ns_dt":pd.to_numeric(single["sum_ns_dt"],errors="coerce"),
        "sum_s_dt":pd.to_numeric(single["sum_s_dt"],errors="coerce"),
        "host_group_source":single["HostGroup"],
        "future_status_source":single["Future_Status_Tooth"],
        "time_to_decay_source":pd.to_numeric(single["Time_to_decay"],errors="coerce"),
        "neighbor_prev_tooth":single["neighbor_prev_tooth"],
        "neighbor_next_tooth":single["neighbor_next_tooth"],
    })
    assert clean.duplicated(["child_id","tooth_position","timepoint"]).sum()==0
    return tooth_map,composite,clean

def build_transitions(clean):
    lookup=clean.set_index(
        ["child_id","timepoint","tooth_position"]
    )[["tooth_status","tooth_dmfs"]]
    s=clean.sort_values(
        ["child_id","tooth_position","time_index","sample_id"]
    ).copy()
    g=s.groupby(["child_id","tooth_position"],sort=False)

    for source,target in [
        ("sample_id","sample_id_t1"),("timepoint","timepoint_t1"),
        ("time_index","time_index_t1"),("age_months","age_months_t1"),
        ("tooth_status","state_t1"),("tooth_state_code","state_code_t1"),
        ("tooth_dmfs","dmfs_tooth_t1"),("host_status","host_status_t1"),
        ("host_dmfs","host_dmfs_t1")
    ]:
        s[target]=g[source].shift(-1)

    obs=s.loc[s["sample_id_t1"].notna()].copy()
    obs["delta_timepoints"]=obs["time_index_t1"]-obs["time_index"]
    obs["delta_months"]=obs["age_months_t1"]-obs["age_months"]
    obs["progression_event"]=(
        (obs["tooth_status"]=="H")&(obs["state_t1"]=="C")
    ).astype(int)
    obs["transition_class"]=obs["tooth_status"]+"->"+obs["state_t1"].astype(str)

    def get_neighbor(row,which,field):
        tooth=row[which]
        if pd.isna(tooth):return np.nan
        try:return lookup.loc[(row["child_id"],row["timepoint"],tooth),field]
        except KeyError:return np.nan

    obs["neighbor_prev_state_t"]=obs.apply(
        lambda r:get_neighbor(r,"neighbor_prev_tooth","tooth_status"),axis=1)
    obs["neighbor_prev_dmfs_t"]=obs.apply(
        lambda r:get_neighbor(r,"neighbor_prev_tooth","tooth_dmfs"),axis=1)
    obs["neighbor_next_state_t"]=obs.apply(
        lambda r:get_neighbor(r,"neighbor_next_tooth","tooth_status"),axis=1)
    obs["neighbor_next_dmfs_t"]=obs.apply(
        lambda r:get_neighbor(r,"neighbor_next_tooth","tooth_dmfs"),axis=1)
    obs["n_neighbors_observed_t"]=obs[
        ["neighbor_prev_state_t","neighbor_next_state_t"]
    ].notna().sum(axis=1)

    trans=pd.DataFrame({
        "transition_id":obs["child_id"]+"|"+obs["tooth_position"]+"|"+
            obs["timepoint"]+"->"+obs["timepoint_t1"].astype(str),
        "child_id":obs["child_id"],"tooth_position":obs["tooth_position"],
        "tooth_fdi":obs["tooth_fdi"],"tooth_type_index":obs["tooth_type_index"],
        "arch":obs["arch"],"arch_index":obs["arch_index"],
        "niche_icm":obs["niche_icm"],"niche_ul":obs["niche_ul"],
        "niche_lr":obs["niche_lr"],"niche_ap":obs["niche_ap"],
        "x":obs["x"],"y":obs["y"],"z":obs["z"],
        "sample_id_t":obs["sample_id"],"timepoint_t":obs["timepoint"],
        "time_index_t":obs["time_index"],"age_months_t":obs["age_months"],
        "state_t":obs["tooth_status"],"state_code_t":obs["tooth_state_code"],
        "dmfs_tooth_t":obs["tooth_dmfs"],"host_status_t":obs["host_status"],
        "host_dmfs_t":obs["host_dmfs"],
        "spatial_weighted_dmfs_t":obs["spatial_weighted_dmfs"],
        "sum_dmfs_t":obs["sum_dmfs"],"sum_ns_dt_t":obs["sum_ns_dt"],
        "sum_s_dt_t":obs["sum_s_dt"],
        "neighbor_prev_tooth":obs["neighbor_prev_tooth"],
        "neighbor_prev_state_t":obs["neighbor_prev_state_t"],
        "neighbor_prev_dmfs_t":obs["neighbor_prev_dmfs_t"],
        "neighbor_next_tooth":obs["neighbor_next_tooth"],
        "neighbor_next_state_t":obs["neighbor_next_state_t"],
        "neighbor_next_dmfs_t":obs["neighbor_next_dmfs_t"],
        "n_neighbors_observed_t":obs["n_neighbors_observed_t"],
        "sample_id_t1":obs["sample_id_t1"],"timepoint_t1":obs["timepoint_t1"],
        "time_index_t1":obs["time_index_t1"],"age_months_t1":obs["age_months_t1"],
        "delta_timepoints":obs["delta_timepoints"],"delta_months":obs["delta_months"],
        "state_t1":obs["state_t1"],"state_code_t1":obs["state_code_t1"],
        "dmfs_tooth_t1":obs["dmfs_tooth_t1"],"host_status_t1":obs["host_status_t1"],
        "host_dmfs_t1":obs["host_dmfs_t1"],"progression_event":obs["progression_event"],
        "transition_class":obs["transition_class"],
        "host_group_source":obs["host_group_source"],
        "future_status_source_t":obs["future_status_source"],
        "time_to_decay_source_t":obs["time_to_decay_source"],
    })
    one=trans.loc[trans["delta_timepoints"].eq(1)].copy()
    onset=one.loc[one["state_t"].eq("H")].copy()
    return trans,one,onset

EXPECTED_COUNTS = {
    "metadata_rows":2504,"children":89,"composite_T5161":220,
    "single_tooth_rows":2284,"all_observed_transitions":1388,
    "one_step_transitions":1160,"H_to_H":913,"H_to_C":84,
    "C_to_C":163,"C_to_H":0,"onset_rows":997,
}

In [ ]:
def add_ca_features(onset):
    ca=onset.copy()
    ca["neighbor_prev_caries"]=(ca["neighbor_prev_state_t"]=="C").astype(int)
    ca["neighbor_next_caries"]=(ca["neighbor_next_state_t"]=="C").astype(int)
    ca["neighbor_caries_count"]=ca["neighbor_prev_caries"]+ca["neighbor_next_caries"]
    return ca

def ca_rule_predict(train,test,prior_strength=20.0):
    global_p=(train["progression_event"].sum()+0.5)/(len(train)+1)

    def table(keys):
        g=train.groupby(keys,dropna=False)["progression_event"].agg(["sum","count"])
        g["p"]=(g["sum"]+prior_strength*global_p)/(g["count"]+prior_strength)
        return g["p"].to_dict()

    t3=table(["niche_icm","host_status_t","neighbor_caries_count"])
    t2=table(["niche_icm","host_status_t"])
    t1=table(["niche_icm"])

    out=[]
    for _,r in test.iterrows():
        k3=(r["niche_icm"],r["host_status_t"],r["neighbor_caries_count"])
        k2=(r["niche_icm"],r["host_status_t"])
        k1=r["niche_icm"]
        out.append(t3.get(k3,t2.get(k2,t1.get(k1,global_p))))
    return np.asarray(out,float)

def safe_group_splits(y,groups,requested):
    tmp=pd.DataFrame({"y":np.asarray(y,int),"g":np.asarray(groups,str)})
    gc=tmp.groupby("g")["y"].max()
    counts=gc.value_counts()
    if len(counts)<2:
        raise ValueError("Grupos positivos/negativos insuficientes.")
    n=min(int(requested),int(counts.min()))
    if n<2:raise ValueError("Menos de 2 folds possíveis.")
    return n

def corrected_group_splits(X, y, groups, n_splits: int, seed: int):
    """Stable child-grouped stratification across affected sklearn versions.

    Numeric group labels are deterministically permuted before invoking
    StratifiedGroupKFold with shuffle=False. This preserves the group-to-label
    mapping and avoids the historical remapping defect observed with
    shuffle=True in older scikit-learn versions.
    """
    _, group_inverse = np.unique(np.asarray(groups), return_inverse=True)
    permutation = np.arange(int(group_inverse.max()) + 1)
    np.random.RandomState(seed).shuffle(permutation)
    inverse_permutation = np.empty_like(permutation)
    inverse_permutation[permutation] = np.arange(permutation.size)
    relabelled = inverse_permutation[group_inverse]
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=False)
    return list(splitter.split(X, y, relabelled))


def ca_oof(ca,requested=5,seed=20260906):
    y=ca["progression_event"].astype(int).to_numpy()
    g=ca["child_id"].astype(str).to_numpy()
    n=safe_group_splits(y,g,requested)
    cv=corrected_group_splits(ca,y,g,n_splits=n,seed=seed)
    pred=np.full(len(ca),np.nan)
    fold=np.full(len(ca),-1)
    for f,(tr,te) in enumerate(cv):
        assert set(g[tr]).isdisjoint(set(g[te]))
        pred[te]=ca_rule_predict(ca.iloc[tr],ca.iloc[te])
        fold[te]=f
    metrics={
        "model":"CA_probabilistic_rule","n":len(ca),"events":int(y.sum()),
        "prevalence":float(y.mean()),
        "PR_AUC":float(average_precision_score(y,pred)),
        "ROC_AUC":float(roc_auc_score(y,pred)),
        "Brier":float(brier_score_loss(y,pred)),
        "LogLoss":float(log_loss(y,np.clip(pred,1e-6,1-1e-6)),
        )
    }
    return metrics,pred,fold

## Primary clinical analysis
The legacy analysis is defined for historical inspection but is not executed. All primary estimates below use the corrected group-split policy.

In [ ]:
"""Exploratory internal validation using the public OdontoCA source cohort.

The two candidate feature sets are fixed before examining model performance.
All preprocessing and regularisation selection occur within child-grouped
training folds. No microbiome or image data are used. Individual records and
identifiers are neither exported nor added to the manuscript directory.
"""

from __future__ import annotations

import hashlib
import json
import platform
from pathlib import Path
from io import BytesIO
import sys
import requests

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, brier_score_loss, log_loss,
    precision_recall_curve, roc_auc_score, roc_curve,
)
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from scipy.optimize import minimize, minimize_scalar
from scipy.special import expit, logit

SOURCE_COMMIT = "e5868fe5664460c7aac1c5b6d7980776ad26b29c"
SOURCE_URL = (
    "https://raw.githubusercontent.com/HuangShiLab/Single-tooth-ECC/"
    + SOURCE_COMMIT + "/Figures_and_Tables_in_Manuscript/Table_S1.xlsx"
)
SOURCE_SHA256 = "b7819fee81efbe2e227b7700c3cd86ce8bc6f7fe6f49da6214f4107d099184fa"


def fit_calibration(y, p):
    score = logit(np.clip(p, 1e-6, 1 - 1e-6))
    offset = minimize_scalar(
        lambda intercept: log_loss(y, expit(intercept + score), normalize=False),
        method="bounded", bounds=(-12, 12),
    )
    line = minimize(
        lambda coef: log_loss(y, expit(coef[0] + coef[1] * score), normalize=False),
        x0=np.array([0.0, 1.0]), method="BFGS",
    )
    if not np.isfinite(line.x).all():
        raise AssertionError("Nonfinite calibration coefficients")
    return {
        "mean_predicted": float(p.mean()),
        "observed_fraction": float(y.mean()),
        "calibration_in_the_large": float(offset.x),
        "calibration_slope": float(line.x[1]),
        "calibration_model_intercept": float(line.x[0]),
    }


OUT = Path(
    "/content/OdontoCA_v1_3_5/09_clinical_addendum"
    if "google.colab" in sys.modules else "./OdontoCA_outputs/09_clinical_addendum"
)
OUT.mkdir(parents=True, exist_ok=True)
SEED = 20260930
BOOTSTRAPS = 1000
FORBIDDEN = {
    "HostGroup", "host_group_source", "Future_Status_Tooth",
    "future_status_source_t", "Time_to_decay", "time_to_decay_source_t",
    "state_t1", "progression_event", "delta_months", "timepoint_t1",
}
FEATURE_SETS = {
    "clinical_minimal": {
        "cat": ["tooth_position"],
        "num": ["age_months_t", "host_dmfs_t", "neighbor_caries_count"],
    },
    "clinical_spatial": {
        "cat": ["tooth_position", "niche_icm", "niche_ul", "niche_lr", "niche_ap", "host_status_t"],
        "num": [
            "age_months_t", "host_dmfs_t", "neighbor_caries_count",
            "spatial_weighted_dmfs_t", "sum_dmfs_t", "sum_ns_dt_t", "sum_s_dt_t",
        ],
    },
}


def source_frame() -> tuple[pd.DataFrame, dict]:
    if not all(name in globals() for name in ("clean_single_tooth", "build_transitions")):
        raise RuntimeError("Execute as células anteriores de reconstrução longitudinal primeiro")
    meta, source_path = load_table_s1()
    content = source_path.read_bytes()
    digest = hashlib.sha256(content).hexdigest()
    if digest != SOURCE_SHA256:
        raise RuntimeError("O hash SHA-256 do suplemento clínico é diferente do auditado")
    meta = pd.read_excel(BytesIO(content), sheet_name="all_metadata")
    meta.columns = [str(col).strip() for col in meta.columns]
    _, _, clean = clean_single_tooth(meta)
    _, _, onset = build_transitions(clean)
    onset = onset.copy()
    onset["neighbor_caries_count"] = (
        onset["neighbor_prev_state_t"].eq("C").astype(int)
        + onset["neighbor_next_state_t"].eq("C").astype(int)
    )
    if len(onset) != 997 or int(onset["progression_event"].sum()) != 84:
        raise AssertionError("As contagens da coorte de origem não foram reproduzidas")
    zero_interval = onset.loc[onset["delta_months"].le(0)]
    frame = onset.loc[onset["delta_months"].gt(0)].reset_index(drop=True)
    if len(frame) != 996 or int(frame["progression_event"].sum()) != 83:
        raise AssertionError("A coorte analítica com seguimento positivo mudou")
    return frame, {
        "source_sha256": digest,
        "reproduced_eligible": 997,
        "reproduced_events": 84,
        "excluded_nonpositive_followup_interval": len(zero_interval),
        "excluded_events": int(zero_interval["progression_event"].sum()),
    }


def build_pipeline(categories: list[str], numeric: list[str]) -> Pipeline:
    try:
        onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        onehot = OneHotEncoder(handle_unknown="ignore", sparse=False)
    prep = ColumnTransformer(
        [
            ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", onehot)]), categories),
            ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric),
        ],
        remainder="drop", sparse_threshold=0,
    )
    return Pipeline([
        ("prep", prep),
        ("model", LogisticRegression(penalty="l2", solver="liblinear", max_iter=5000)),
    ])


def interval(values: list[float]) -> list[float]:
    return [float(v) for v in np.quantile(values, [0.025, 0.975])]


def run() -> None:
    frame, source = source_frame()
    y = frame["progression_event"].to_numpy(dtype=int)
    groups = frame["child_id"].astype(str).to_numpy()
    for definition in FEATURE_SETS.values():
        if FORBIDDEN.intersection(definition["cat"] + definition["num"]):
            raise AssertionError("A future or outcome-derived variable entered the model")

    predictions = {name: np.full(len(frame), np.nan) for name in FEATURE_SETS}
    prevalence_reference = np.full(len(frame), np.nan)
    fold_assignment = np.full(len(frame), -1)
    fold_log = []
    outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    for fold, (train, test) in enumerate(outer.split(frame, y, groups)):
        if not set(groups[train]).isdisjoint(groups[test]):
            raise AssertionError("Child overlap across outer folds")
        if len(np.unique(y[test])) != 2:
            raise AssertionError("Outer test fold lacks one outcome class")
        prevalence_reference[test] = y[train].mean()
        inner = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED + 100 + fold)
        log = {
            "fold": fold,
            "train_rows": len(train), "test_rows": len(test),
            "train_children": len(set(groups[train])), "test_children": len(set(groups[test])),
            "train_events": int(y[train].sum()), "test_events": int(y[test].sum()),
        }
        for name, definition in FEATURE_SETS.items():
            columns = definition["cat"] + definition["num"]
            search = GridSearchCV(
                build_pipeline(definition["cat"], definition["num"]),
                {"model__C": [0.01, 0.1, 1.0, 10.0]},
                scoring="average_precision", cv=inner, n_jobs=1,
                refit=True, error_score="raise",
            )
            search.fit(frame.iloc[train][columns], y[train], groups=groups[train])
            classes = search.best_estimator_.named_steps["model"].classes_
            positive = np.flatnonzero(classes == 1)
            if len(positive) != 1:
                raise AssertionError("Positive class is absent or ambiguous")
            predictions[name][test] = search.predict_proba(frame.iloc[test][columns])[:, int(positive[0])]
            log[name + "_C"] = search.best_params_["model__C"]
        fold_assignment[test] = fold
        fold_log.append(log)

    if pd.DataFrame({"child": groups, "fold": fold_assignment}).groupby("child")["fold"].nunique().max() != 1:
        raise AssertionError("Grouped validation invariant failed")
    if not all(np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all() for p in predictions.values()):
        raise AssertionError("Invalid out-of-fold probability")

    metrics = {}
    for name, p in predictions.items():
        metrics[name] = {
            "pr_auc": float(average_precision_score(y, p)),
            "roc_auc": float(roc_auc_score(y, p)),
            "brier": float(brier_score_loss(y, p)),
            "log_loss": float(log_loss(y, np.clip(p, 1e-6, 1 - 1e-6))),
            **fit_calibration(y, p),
        }

    rng = np.random.default_rng(SEED + 1)
    children = np.unique(groups)
    child_indices = {child: np.flatnonzero(groups == child) for child in children}
    sampled_metrics = {name: {metric: [] for metric in ["pr_auc", "roc_auc", "brier"]} for name in predictions}
    sampled_deltas = {metric: [] for metric in ["pr_auc", "roc_auc", "brier"]}
    valid = 0
    for _ in range(BOOTSTRAPS):
        sampled = rng.choice(children, size=len(children), replace=True)
        index = np.concatenate([child_indices[child] for child in sampled])
        yy = y[index]
        if len(np.unique(yy)) != 2:
            continue
        valid += 1
        for name, p in predictions.items():
            pp = p[index]
            sampled_metrics[name]["pr_auc"].append(float(average_precision_score(yy, pp)))
            sampled_metrics[name]["roc_auc"].append(float(roc_auc_score(yy, pp)))
            sampled_metrics[name]["brier"].append(float(brier_score_loss(yy, pp)))
        for metric in sampled_deltas:
            sampled_deltas[metric].append(
                sampled_metrics["clinical_spatial"][metric][-1]
                - sampled_metrics["clinical_minimal"][metric][-1]
            )
    for name in metrics:
        for metric, values in sampled_metrics[name].items():
            metrics[name][metric + "_ci95"] = interval(values)
    differences = {
        metric: {
            "estimate": metrics["clinical_spatial"][metric] - metrics["clinical_minimal"][metric],
            "ci95": interval(values),
        }
        for metric, values in sampled_deltas.items()
    }

    report = {
        "analysis_status": "EXPLORATORY_INTERNAL_VALIDATION_ONLY",
        "source": source,
        "source_commit": SOURCE_COMMIT,
                "runtime": {
            "python": platform.python_version(),
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "scipy": scipy.__version__,
            "scikit_learn": sklearn.__version__,
        },
        "analysis_rows": len(frame),
        "analysis_events": int(y.sum()),
        "analysis_children": len(children),
        "event_children": int(frame.loc[frame["progression_event"].eq(1), "child_id"].nunique()),
        "event_fraction": float(y.mean()),
        "followup_months_median": float(frame["delta_months"].median()),
        "followup_months_iqr": [float(v) for v in frame["delta_months"].quantile([0.25, 0.75])],
        "followup_months_range": [float(frame["delta_months"].min()), float(frame["delta_months"].max())],
        "seed": SEED, "outer_folds": 5, "inner_folds": 3,
        "feature_sets": FEATURE_SETS,
        "fold_log": fold_log,
        "child_disjoint_outer_folds": True,
        "bootstrap_requested": BOOTSTRAPS, "bootstrap_valid": valid,
        "metrics": metrics,
        "prevalence_reference": {
            "brier": float(brier_score_loss(y, prevalence_reference)),
            "log_loss": float(log_loss(y, prevalence_reference)),
        },
        "spatial_minus_minimal": differences,
        "scope_note": (
            "This is internal validation in one public cohort with no external test, "
            "real microbiome alignment or image branch. Patient use is unsupported."
        ),
    }
    (OUT / "real_clinical_internal_validation.json").write_text(json.dumps(report, ensure_ascii=False, indent=2))

    plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2), dpi=170)
    colors = {"clinical_minimal": "#25668c", "clinical_spatial": "#bf7924"}
    for name, p in predictions.items():
        precision, recall, _ = precision_recall_curve(y, p)
        fpr, tpr, _ = roc_curve(y, p)
        label = "Minimal clinical" if name == "clinical_minimal" else "Clinical + spatial"
        axes[0].plot(recall, precision, label=f"{label} (AP {metrics[name]['pr_auc']:.3f})", color=colors[name], lw=2)
        axes[1].plot(fpr, tpr, label=f"{label} (AUC {metrics[name]['roc_auc']:.3f})", color=colors[name], lw=2)
    axes[0].axhline(y.mean(), color="#777777", ls="--", label=f"Event fraction {y.mean():.3f}")
    axes[1].plot([0, 1], [0, 1], color="#777777", ls="--", label="Chance")
    axes[0].set(xlabel="Recall", ylabel="Precision", title="A. Precision–recall")
    axes[1].set(xlabel="False-positive rate", ylabel="True-positive rate", title="B. ROC")
    for ax in axes:
        ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.legend(loc="best", frameon=False, fontsize=8)
    fig.suptitle("Public clinical cohort: child-grouped internal out-of-fold predictions", fontsize=12)
    fig.tight_layout()
    fig.savefig(OUT / "Figure_2_real_internal_validation.png", bbox_inches="tight")
    fig.savefig(OUT / "Figure_2_real_internal_validation.pdf", bbox_inches="tight")
    plt.close(fig)

    # Descriptive calibration only: each point pools one fifth of observations
    # ordered by out-of-fold predicted probability, without outcome fitting.
    fig, ax = plt.subplots(figsize=(5.5, 5.0), dpi=170)
    max_bin_value = 0.0
    for name, p in predictions.items():
        ordered = np.argsort(p)
        bins = np.array_split(ordered, 5)
        mean_pred = [float(p[part].mean()) for part in bins]
        observed = [float(y[part].mean()) for part in bins]
        max_bin_value = max(max_bin_value, *mean_pred, *observed)
        label = "Minimal clinical" if name == "clinical_minimal" else "Clinical + spatial"
        ax.plot(mean_pred, observed, marker="o", color=colors[name], lw=2, label=label)
    axis_max = min(1.0, max_bin_value * 1.12)
    ax.plot([0, axis_max], [0, axis_max], ls="--", color="#777777", label="Ideal calibration")
    ax.set(xlim=(0, axis_max), ylim=(0, axis_max), xlabel="Mean predicted risk", ylabel="Observed event fraction",
           title="Internal calibration, five equal-count bins")
    ax.legend(loc="upper left", frameon=False, fontsize=8)
    fig.subplots_adjust(left=0.19, right=0.98, top=0.93, bottom=0.15)
    fig.savefig(OUT / "Figure_3_real_internal_calibration.png", bbox_inches="tight")
    fig.savefig(OUT / "Figure_3_real_internal_calibration.pdf", bbox_inches="tight")
    plt.close(fig)

    print(json.dumps({
        "analysis_rows": report["analysis_rows"],
        "analysis_events": report["analysis_events"],
        "analysis_children": report["analysis_children"],
        "event_fraction": report["event_fraction"],
        "metrics": metrics,
        "prevalence_reference": report["prevalence_reference"],
        "spatial_minus_minimal": differences,
    }, ensure_ascii=False, indent=2))


if bool(globals().get("RUN_LEGACY_SPLIT_UI", False)):
    LEGACY_INTERNAL_VALIDATION = run()
else:
    print("Análise legada não executada (RUN_LEGACY_SPLIT_UI=False).")


In [ ]:
"""Nested, child-grouped evaluation of train-only probability recalibration.

The corrected group permutation policy is fixed explicitly and does not
depend on the historical shuffle=True implementation. Only
aggregate statistics and figures are written; no individual predictions or
patient identifiers leave this process.
"""

from __future__ import annotations

import json
import platform
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy.optimize import minimize, minimize_scalar
from scipy.special import expit, logit
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold

# Cohort reconstruction and base-model functions are supplied by the preceding cell.


OUT = Path("/content/OdontoCA_v1_3_5/10_calibracao" if "google.colab" in sys.modules else "./OdontoCA_outputs/10_calibracao")
OUT.mkdir(parents=True, exist_ok=True)
METHODS = ("uncalibrated", "intercept_only", "intercept_and_slope")
VERSION_TAG = sklearn.__version__.replace(".", "")
SPLIT_POLICY = "corrected_permuted_groups"


def corrected_group_splits(X, y, groups, n_splits: int, seed: int):
    """Reproduce the corrected shuffle algorithm across sklearn versions.

    Permuting numeric group labels before shuffle=False yields the same stable
    tie order as the corrected shuffle=True implementation, while retaining
    the label-to-group mapping in affected older versions.
    """
    _, group_inverse = np.unique(groups, return_inverse=True)
    permutation = np.arange(int(group_inverse.max()) + 1)
    np.random.RandomState(seed).shuffle(permutation)
    inverse_permutation = np.empty_like(permutation)
    inverse_permutation[permutation] = np.arange(permutation.size)
    relabelled = inverse_permutation[group_inverse]
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=False)
    return list(splitter.split(X, y, relabelled))


def _score(p: np.ndarray) -> np.ndarray:
    return logit(np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6))


def fit_recalibrators(y: np.ndarray, p: np.ndarray) -> dict[str, dict[str, float]]:
    """Fit two monotone maps using training-only cross-fitted probabilities."""
    y = np.asarray(y, dtype=int)
    x = _score(p)
    if len(np.unique(y)) != 2 or not np.isfinite(x).all():
        raise ValueError("Calibration training data need both outcomes and finite scores")
    offset = minimize_scalar(
        lambda a: log_loss(y, expit(a + x), normalize=False),
        method="bounded", bounds=(-12, 12),
    )
    if not offset.success:
        raise RuntimeError("Intercept recalibration failed")
    # The positive slope constraint keeps the map monotone in risk.
    line = minimize(
        lambda ab: log_loss(y, expit(ab[0] + ab[1] * x), normalize=False),
        x0=np.array([float(offset.x), 1.0]),
        method="L-BFGS-B", bounds=[(-12, 12), (0.05, 3.0)],
    )
    if not line.success or not np.isfinite(line.x).all():
        raise RuntimeError("Intercept-and-slope recalibration failed")
    return {
        "intercept_only": {"intercept": float(offset.x), "slope": 1.0},
        "intercept_and_slope": {
            "intercept": float(line.x[0]), "slope": float(line.x[1]),
        },
    }


def apply_recalibration(p: np.ndarray, parameters: dict[str, float]) -> np.ndarray:
    return expit(parameters["intercept"] + parameters["slope"] * _score(p))


def _metrics(y: np.ndarray, p: np.ndarray) -> dict[str, float]:
    return {
        "pr_auc": float(average_precision_score(y, p)),
        "roc_auc": float(roc_auc_score(y, p)),
        "brier": float(brier_score_loss(y, p)),
        "log_loss": float(log_loss(y, np.clip(p, 1e-6, 1 - 1e-6))),
        **fit_calibration(y, p),
    }


def _reliability(ax, y: np.ndarray, p: np.ndarray, label: str, color: str) -> None:
    boundaries = np.unique(np.quantile(p, np.linspace(0, 1, 9)))
    if len(boundaries) < 3:
        return
    bins = np.digitize(p, boundaries[1:-1], right=True)
    px = [p[bins == k].mean() for k in range(len(boundaries) - 1) if (bins == k).any()]
    py = [y[bins == k].mean() for k in range(len(boundaries) - 1) if (bins == k).any()]
    ax.plot(px, py, "o-", lw=1.5, ms=4, label=label, color=color)


def _figures(y: np.ndarray, predictions: dict[str, dict[str, np.ndarray]], metrics: dict) -> None:
    colors = {"clinical_minimal": "#25668c", "clinical_spatial": "#bf7924"}
    labels = {"clinical_minimal": "Minimal clinical", "clinical_spatial": "Clinical + spatial"}
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2), dpi=600)
    for name, variants in predictions.items():
        p = variants["uncalibrated"]
        precision, recall, _ = precision_recall_curve(y, p)
        fpr, tpr, _ = roc_curve(y, p)
        axes[0].plot(recall, precision, label=f"{labels[name]} (AP {metrics[name]['uncalibrated']['pr_auc']:.3f})", color=colors[name], lw=2)
        axes[1].plot(fpr, tpr, label=f"{labels[name]} (AUC {metrics[name]['uncalibrated']['roc_auc']:.3f})", color=colors[name], lw=2)
    axes[0].axhline(y.mean(), ls="--", color="#777", label=f"Event fraction {y.mean():.3f}")
    axes[1].plot([0, 1], [0, 1], ls="--", color="#777", label="Chance")
    axes[0].set(xlabel="Recall", ylabel="Precision", title="A. Precision–recall")
    axes[1].set(xlabel="False-positive rate", ylabel="True-positive rate", title="B. ROC")
    for ax in axes:
        ax.legend(fontsize=8)
        ax.grid(alpha=0.15)
    fig.tight_layout()
    fig.savefig(OUT / f"Figure_2_discrimination_corrected_sklearn{VERSION_TAG}.png", bbox_inches="tight")
    fig.savefig(OUT / f"Figure_2_discrimination_corrected_sklearn{VERSION_TAG}.pdf", bbox_inches="tight")
    fig.savefig(OUT / f"Figure_2_discrimination_corrected_sklearn{VERSION_TAG}.svg", bbox_inches="tight")
    plt.close(fig)

    fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2), dpi=600)
    method_labels = {
        "uncalibrated": "Original", "intercept_only": "Intercept updated",
        "intercept_and_slope": "Intercept and slope updated",
    }
    method_colors = {
        "uncalibrated": "#69747c", "intercept_only": "#25668c",
        "intercept_and_slope": "#bf7924",
    }
    for ax, (name, variants) in zip(axes, predictions.items()):
        ax.plot([0, 0.5], [0, 0.5], ls="--", color="#333", lw=1, label="Ideal")
        for method, p in variants.items():
            _reliability(ax, y, p, method_labels[method], method_colors[method])
        ax.set(xlim=(0, 0.5), ylim=(0, 0.5), xlabel="Mean predicted risk", ylabel="Observed event fraction", title=labels[name])
        ax.grid(alpha=0.15)
        ax.legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(OUT / f"Figure_3_calibration_corrected_sklearn{VERSION_TAG}.png", bbox_inches="tight")
    fig.savefig(OUT / f"Figure_3_calibration_corrected_sklearn{VERSION_TAG}.pdf", bbox_inches="tight")
    fig.savefig(OUT / f"Figure_3_calibration_corrected_sklearn{VERSION_TAG}.svg", bbox_inches="tight")
    plt.close(fig)


def run_calibrated() -> dict:
    frame, source = source_frame()
    y = frame["progression_event"].to_numpy(dtype=int)
    groups = frame["child_id"].astype(str).to_numpy()
    if any(FORBIDDEN.intersection(d["cat"] + d["num"]) for d in FEATURE_SETS.values()):
        raise AssertionError("Future/outcome-derived variable entered the model")
    predictions = {
        name: {method: np.full(len(y), np.nan) for method in METHODS}
        for name in FEATURE_SETS
    }
    fold_ids = np.full(len(y), -1)
    folds = []
    prevalence = np.full(len(y), np.nan)
    outer_splits = corrected_group_splits(frame, y, groups, n_splits=5, seed=SEED)
    for fold, (train, test) in enumerate(outer_splits):
        if not set(groups[train]).isdisjoint(groups[test]) or len(np.unique(y[test])) != 2:
            raise AssertionError("Outer child separation or outcome balance failed")
        prevalence[test] = y[train].mean()
        fold_ids[test] = fold
        detail = {
            "fold": fold, "train_rows": len(train), "test_rows": len(test),
            "train_children": len(set(groups[train])), "test_children": len(set(groups[test])),
            "train_events": int(y[train].sum()), "test_events": int(y[test].sum()),
        }
        splits = corrected_group_splits(
            frame.iloc[train], y[train], groups[train],
            n_splits=3, seed=SEED + 100 + fold,
        )
        if any(not set(groups[train][a]).isdisjoint(groups[train][b]) for a, b in splits):
            raise AssertionError("Inner child separation failed")
        for name, definition in FEATURE_SETS.items():
            columns = definition["cat"] + definition["num"]
            xtrain, xtest = frame.iloc[train][columns], frame.iloc[test][columns]
            search = GridSearchCV(
                build_pipeline(definition["cat"], definition["num"]),
                {"model__C": [0.01, 0.1, 1.0, 10.0]},
                scoring="average_precision", cv=splits, n_jobs=1,
                refit=True, error_score="raise",
            )
            search.fit(xtrain, y[train])
            chosen_c = search.best_params_["model__C"]
            positive = np.flatnonzero(search.best_estimator_.classes_ == 1)
            if len(positive) != 1:
                raise AssertionError("Positive outcome class is ambiguous")
            raw_test = search.predict_proba(xtest)[:, int(positive[0])]
            predictions[name]["uncalibrated"][test] = raw_test

            # Calibration sees only cross-fitted predictions from this outer TRAIN.
            inner_oof = np.full(len(train), np.nan)
            for a, b in splits:
                model = build_pipeline(definition["cat"], definition["num"])
                model.set_params(model__C=chosen_c)
                model.fit(xtrain.iloc[a], y[train][a])
                positive_inner = np.flatnonzero(model.classes_ == 1)
                if len(positive_inner) != 1:
                    raise AssertionError("Inner positive class is ambiguous")
                inner_oof[b] = model.predict_proba(xtrain.iloc[b])[:, int(positive_inner[0])]
            if not np.isfinite(inner_oof).all():
                raise AssertionError("Incomplete inner cross-fitting")
            recalibrators = fit_recalibrators(y[train], inner_oof)
            for method, params in recalibrators.items():
                predictions[name][method][test] = apply_recalibration(raw_test, params)
            detail[name + "_C"] = chosen_c
            detail[name + "_calibration"] = recalibrators
        folds.append(detail)

    if (fold_ids < 0).any() or pd.DataFrame({"child": groups, "fold": fold_ids}).groupby("child")["fold"].nunique().max() != 1:
        raise AssertionError("Outer grouped OOF coverage failed")
    if any(not np.isfinite(p).all() or ((p < 0) | (p > 1)).any() for v in predictions.values() for p in v.values()):
        raise AssertionError("Invalid external-fold probability")
    metrics = {name: {method: _metrics(y, p) for method, p in variants.items()} for name, variants in predictions.items()}

    # Paired child bootstrap conditions on the already cross-fitted predictions.
    rng = np.random.default_rng(SEED + 1)
    children = np.unique(groups)
    child_indices = {child: np.flatnonzero(groups == child) for child in children}
    differences = {
        name: {method: {metric: [] for metric in ("brier", "log_loss")}
               for method in METHODS if method != "uncalibrated"}
        for name in FEATURE_SETS
    }
    raw_samples = {name:{metric:[] for metric in ("roc_auc","pr_auc","brier")} for name in FEATURE_SETS}
    model_deltas = {metric:[] for metric in ("roc_auc","pr_auc","brier")}
    valid = 0
    for _ in range(BOOTSTRAPS):
        sampled = rng.choice(children, size=len(children), replace=True)
        index = np.concatenate([child_indices[child] for child in sampled])
        yy = y[index]
        if len(np.unique(yy)) != 2:
            continue
        valid += 1
        for name, variants in predictions.items():
            raw = variants["uncalibrated"][index]
            raw_samples[name]["roc_auc"].append(float(roc_auc_score(yy,raw)))
            raw_samples[name]["pr_auc"].append(float(average_precision_score(yy,raw)))
            raw_samples[name]["brier"].append(float(brier_score_loss(yy,raw)))
        for metric in model_deltas:
            model_deltas[metric].append(raw_samples["clinical_spatial"][metric][-1]-raw_samples["clinical_minimal"][metric][-1])
        for name, variants in predictions.items():
            raw = variants["uncalibrated"][index]
            for method in differences[name]:
                calibrated = variants[method][index]
                differences[name][method]["brier"].append(float(brier_score_loss(yy, calibrated) - brier_score_loss(yy, raw)))
                differences[name][method]["log_loss"].append(float(log_loss(yy, calibrated) - log_loss(yy, raw)))
    delta_report = {
        name: {
            method: {
                metric: {
                    "estimate": metrics[name][method][metric] - metrics[name]["uncalibrated"][metric],
                    "ci95": interval(values),
                }
                for metric, values in by_metric.items()
            }
            for method, by_metric in by_method.items()
        }
        for name, by_method in differences.items()
    }
    report = {
        "analysis_status": "EXPLORATORY_INTERNAL_VALIDATION_ONLY",
        "source": source,
        "runtime": {
            "python": platform.python_version(), "numpy": np.__version__,
            "pandas": pd.__version__, "scipy": scipy.__version__,
            "scikit_learn": sklearn.__version__,
        },
        "analysis_rows": len(y), "analysis_events": int(y.sum()),
        "analysis_children": len(children),
        "event_children": len(np.unique(groups[y == 1])),
        "event_fraction": float(y.mean()),
        "seed": SEED, "outer_folds": 5, "inner_folds": 3,
        "split_policy": SPLIT_POLICY,
        "fold_log": folds,
        "metrics": metrics,
        "prevalence_reference": {
            "brier": float(brier_score_loss(y, prevalence)),
            "log_loss": float(log_loss(y, prevalence)),
        },
        "calibration_minus_uncalibrated": delta_report,
        "uncalibrated_ci95": {name:{metric:interval(values) for metric,values in by_metric.items()} for name,by_metric in raw_samples.items()},
        "spatial_minus_minimal": {metric:{"estimate":metrics["clinical_spatial"]["uncalibrated"][metric]-metrics["clinical_minimal"]["uncalibrated"][metric],"ci95":interval(values)} for metric,values in model_deltas.items()},
        "bootstrap_requested": BOOTSTRAPS, "bootstrap_valid": valid,
        "scope_note": "Cross-fitted internal calibration within 81 children; no external validation or clinical use.",
    }
    path = OUT / f"calibrated_clinical_internal_validation_corrected_sklearn{VERSION_TAG}.json"
    path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    _figures(y, predictions, metrics)
    print(json.dumps({"metrics": metrics, "calibration_minus_uncalibrated": delta_report}, ensure_ascii=False, indent=2))
    return report


if bool(globals().get("RUN_PRIMARY_ANALYSIS_UI", True)):
    PRIMARY_INTERNAL_VALIDATION = run_calibrated()
else:
    print("Análise primária não executada (RUN_PRIMARY_ANALYSIS_UI=False).")


In [ ]:
frame, source = source_frame()
REAL = {"ca": frame}


## Figure 5
Native vector anatomy schematic and real source coordinates. Circle area is exactly proportional to eligible transition counts. Zero events are descriptive, not proof of zero risk.

In [ ]:
# 25.1 — Figure 5 do manuscrito: exportação nativa 600 dpi + SVG/PDF
# Requer PROFILE_UI = "CLINICAL_INTERNAL", "REPRO_DATASET" ou "FULL_RESEARCH"
# e a execução prévia da seção 16 (REAL["ca"]).

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import Normalize
from matplotlib.patches import Ellipse, Arc, FancyBboxPatch

FIG5_DPI = 600
FIG5_W_IN = 5.8
FIG5_H_IN = 4.35
FIG5_EXPECTED_PX = (3480, 2610)

if REAL.get("ca") is None:
    raise RuntimeError(
        "Figure 5 requer a coorte clínica real. Use PROFILE_UI='CLINICAL_INTERNAL' "
        "ou outro perfil que carregue Table_S1 e execute a seção 16."
    )

# Mesma coorte primária do manuscrito: H em t, próxima visita consecutiva e delta_months > 0.
fig5_frame = REAL["ca"].loc[REAL["ca"]["delta_months"].gt(0)].copy()
fig5_frame["progression_event"] = fig5_frame["progression_event"].astype(int)

assert len(fig5_frame) == 996, f"Esperado 996; observado {len(fig5_frame)}"
assert int(fig5_frame["progression_event"].sum()) == 83
assert fig5_frame["child_id"].nunique() == 81

# Dados agregados reproduzíveis usados no painel (a) e na escala cromática.
fig5_stats = (
    fig5_frame.groupby("tooth_position", observed=False)
    .agg(
        transitions=("progression_event", "size"),
        events=("progression_event", "sum"),
        x=("x", "mean"),
        y=("y", "mean"),
    )
    .reindex(ALL_PRIMARY)
    .reset_index()
)
fig5_stats["event_fraction"] = fig5_stats["events"] / fig5_stats["transitions"]

EXPECTED_FIG5 = {
    "T51": (12, 0), "T52": (16, 0), "T53": (16, 0), "T54": (98, 3), "T55": (122, 14),
    "T61": (12, 0), "T62": (16, 0), "T63": (16, 0), "T64": (106, 3), "T65": (119, 15),
    "T71": (14, 0), "T72": (16, 0), "T73": (16, 0), "T74": (94, 10), "T75": (91, 15),
    "T81": (15, 0), "T82": (16, 0), "T83": (16, 0), "T84": (88, 8), "T85": (97, 15),
}
observed_counts = {
    r.tooth_position: (int(r.transitions), int(r.events))
    for r in fig5_stats.itertuples()
}
assert observed_counts == EXPECTED_FIG5, (
    "As contagens por dente mudaram; interrompendo exportação para evitar figura incorreta."
)

# -------------------------- layout do painel (a) --------------------------
# Centros dos dentes (esquema vetorial odontológico, não radiografia).
UPPER_TOOTH = {
    "T55":(-0.72,0.38), "T54":(-0.57,0.59), "T53":(-0.40,0.79),
    "T52":(-0.23,0.93), "T51":(-0.08,1.00),
    "T61":( 0.08,1.00), "T62":( 0.23,0.93), "T63":( 0.40,0.79),
    "T64":( 0.57,0.59), "T65":( 0.72,0.38),
}
LOWER_TOOTH = {
    "T85":(-0.72,-0.43), "T84":(-0.57,-0.63), "T83":(-0.40,-0.82),
    "T82":(-0.23,-0.95), "T81":(-0.08,-1.02),
    "T71":( 0.08,-1.02), "T72":( 0.23,-0.95), "T73":( 0.40,-0.82),
    "T74":( 0.57,-0.63), "T75":( 0.72,-0.43),
}

# Marcadores externos dos dados. T82/T81/T71/T72 ficam abaixo da arcada,
# formando o arco inferior solicitado.
UPPER_MARKER = {
    "T55":(-0.96,0.42), "T54":(-0.86,0.69), "T53":(-0.66,0.98),
    "T52":(-0.39,1.17), "T51":(-0.14,1.28),
    "T61":( 0.14,1.28), "T62":( 0.39,1.17), "T63":( 0.66,0.98),
    "T64":( 0.86,0.69), "T65":( 0.96,0.42),
}
LOWER_MARKER = {
    "T85":(-0.93,-1.20), "T84":(-0.80,-0.94), "T83":(-0.53,-0.80),
    "T82":(-0.31,-1.31), "T81":(-0.11,-1.43),
    "T71":( 0.11,-1.43), "T72":( 0.31,-1.31),
    "T73":( 0.53,-0.80), "T74":( 0.80,-0.94), "T75":( 0.93,-1.20),
}

stats = fig5_stats.set_index("tooth_position")
max_n = float(stats["transitions"].max())
rate_max = max(0.165, float(stats["event_fraction"].max()))
norm = Normalize(vmin=0.0, vmax=rate_max)
cmap = plt.get_cmap("YlOrRd")

def _draw_tooth(ax, x, y, tooth, upper=True):
    # Esquema de coroa dental em vetor. Molares > caninos > incisivos.
    if tooth in {"T55","T54","T64","T65","T85","T84","T74","T75"}:
        w,h = 0.19,0.16
    elif tooth in {"T53","T63","T83","T73"}:
        w,h = 0.135,0.17
    else:
        w,h = 0.12,0.16
    angle = np.degrees(np.arctan2(y - (0.55 if upper else -0.55), x)) * 0.12
    crown = Ellipse(
        (x,y), width=w, height=h, angle=angle,
        facecolor="#fffdf7", edgecolor="#b9a99d", lw=0.65, zorder=4
    )
    ax.add_patch(crown)
    # sulco discreto
    ax.plot([x-w*0.22,x+w*0.22],[y,y], color="#ded4cd", lw=0.35, zorder=5)

def _draw_arch(ax):
    # gengiva/palato esquemáticos
    ax.add_patch(Ellipse((0,0.62),1.58,1.18,facecolor="#f8d3d0",
                         edgecolor="none",alpha=0.65,zorder=0))
    ax.add_patch(Ellipse((0,-0.65),1.58,1.10,facecolor="#f8d3d0",
                         edgecolor="none",alpha=0.65,zorder=0))
    ax.add_patch(Ellipse((0,0.61),1.18,0.75,facecolor="#fce8e5",
                         edgecolor="none",alpha=0.80,zorder=1))
    ax.add_patch(Ellipse((0,-0.67),1.08,0.64,facecolor="#f8b8b3",
                         edgecolor="none",alpha=0.55,zorder=1))
    ax.add_patch(Arc((0,0.58),1.48,1.00,theta1=20,theta2=160,
                     color="#cf8d88",lw=1.2,zorder=2))
    ax.add_patch(Arc((0,-0.61),1.48,0.96,theta1=200,theta2=340,
                     color="#cf8d88",lw=1.2,zorder=2))
    for tooth,(x,y) in UPPER_TOOTH.items():
        _draw_tooth(ax,x,y,tooth,upper=True)
    for tooth,(x,y) in LOWER_TOOTH.items():
        _draw_tooth(ax,x,y,tooth,upper=False)

def _marker_size(n):
    # scatter usa área; portanto s ∝ n equivale a raio ∝ sqrt(n)
    return 159.0 * (float(n) / max_n)

def _plot_marker(ax, tooth, xy, lower=False):
    row = stats.loc[tooth]
    n = int(row["transitions"])
    ev = int(row["events"])
    rate = float(row["event_fraction"])
    x,y = xy
    edge = "#a51f13" if ev > 0 else "#60778a"
    lw = 1.25 if ev > 0 else 0.55
    ax.scatter([x],[y],s=_marker_size(n),c=[cmap(norm(rate))],
               edgecolors=edge,linewidths=lw,zorder=7)

    # Rótulos: anterior inferior aparece ABAIXO dos círculos.
    if not lower and tooth in {"T51","T52","T61","T62"}:
        ax.text(x,y+0.15,f"{tooth}\n{n} / {ev}",ha="center",va="bottom",fontsize=5.1,color="#263b55",zorder=8)
    elif lower and tooth in {"T82","T81","T71","T72"}:
        ty = y - 0.095
        va = "top"
        ax.text(x,ty,tooth,ha="center",va=va,fontsize=5.1,color="#263b55",zorder=8)
        ax.text(x,ty-0.075,f"{n}",ha="center",va=va,fontsize=5.0,
                color="#172436",fontweight="bold",zorder=8)
        ax.text(x,ty-0.145,f"{ev}",ha="center",va=va,fontsize=5.0,
                color="#b21f18",fontweight="bold",zorder=8)
    else:
        # Laterais: rótulo deslocado para fora da arcada.
        side = -1 if x < 0 else 1
        tx = x + side*0.12
        ha = "right" if side < 0 else "left"
        ax.text(tx,y+0.055,tooth,ha=ha,va="bottom",fontsize=5.1,color="#263b55",zorder=8)
        ax.text(tx,y-0.005,f"{n}",ha=ha,va="center",fontsize=5.0,
                color="#172436",fontweight="bold",zorder=8)
        ax.text(tx,y-0.070,f"{ev}",ha=ha,va="top",fontsize=5.0,
                color="#b21f18",fontweight="bold",zorder=8)

# -------------------------- construção da figura --------------------------
fig = plt.figure(figsize=(FIG5_W_IN, FIG5_H_IN), dpi=FIG5_DPI, facecolor="white")
gs = fig.add_gridspec(
    nrows=2, ncols=2, height_ratios=[0.93, 0.07], width_ratios=[1.08, 0.92],
    hspace=0.03, wspace=0.06, left=0.025, right=0.90, top=0.86, bottom=0.07
)
ax_a = fig.add_subplot(gs[0,0])
ax_b = fig.add_subplot(gs[0,1])
ax_note = fig.add_subplot(gs[1,:])

fig.suptitle(
    "Tooth transitions and observed H→C events\n"
    "(996 transitions, 83 events, 81 children)",
    x=0.025, y=0.975, ha="left", fontsize=7.4, fontweight="bold", color="#14263a"
)

for ax in (ax_a, ax_b):
    ax.set_facecolor("white")
    for spine in ax.spines.values():
        spine.set_color("#a9cbe5")
        spine.set_linewidth(0.75)

ax_a.set_title(
    "(a) Primary dentition: transitions and events",
    loc="left", fontsize=6.0, fontweight="bold", pad=4, color="#14263a"
)
_draw_arch(ax_a)
for t,xy in UPPER_MARKER.items():
    _plot_marker(ax_a,t,xy,lower=False)
for t,xy in LOWER_MARKER.items():
    _plot_marker(ax_a,t,xy,lower=True)

# Faixas de identificação
for y,label in [(0.22,"Upper primary dentition"),(-0.05,"Lower primary dentition")]:
    box = FancyBboxPatch(
        (-0.34,y-0.055),0.68,0.11,boxstyle="round,pad=0.02,rounding_size=0.04",
        facecolor="#e7f1fb",edgecolor="none",zorder=9
    )
    ax_a.add_patch(box)
    ax_a.text(0,y,label,ha="center",va="center",fontsize=5.8,
              fontweight="bold",color="#20364d",zorder=10)

ax_a.set_xlim(-1.18,1.18)
ax_a.set_ylim(-1.86,1.70)
ax_a.set_xticks([]); ax_a.set_yticks([])
ax_a.set_aspect("equal", adjustable="box")

# -------------------------- painel (b): coordenadas reais --------------------------
ax_b.set_title(
    "(b) Observed events in source coordinates",
    loc="left", fontsize=6.0, fontweight="bold", pad=4, color="#14263a"
)

# Trabalha diretamente com as coordenadas registradas no dataset.
x = pd.to_numeric(fig5_frame["x"], errors="coerce").to_numpy(float)
y = pd.to_numeric(fig5_frame["y"], errors="coerce").to_numpy(float)
evt = fig5_frame["progression_event"].to_numpy(int)
valid = np.isfinite(x) & np.isfinite(y)

# Silhueta neutra de referência; não é dado anatômico.
if valid.any():
    cx, cy = np.nanmedian(x[valid]), np.nanmedian(y[valid])
    rx = max(np.nanpercentile(x[valid],95)-np.nanpercentile(x[valid],5), 1e-6)
    ry = max(np.nanpercentile(y[valid],95)-np.nanpercentile(y[valid],5), 1e-6)
    tongue = Ellipse(
        (cx, cy - 0.08*ry), width=0.35*rx, height=0.63*ry,
        facecolor="#fbe7e7", edgecolor="#f2a9a9", lw=0.65, ls="--", zorder=0
    )
    ax_b.add_patch(tongue)

ax_b.scatter(x[valid & (evt==0)], y[valid & (evt==0)],
             s=4.0, c="#cbd5dd", edgecolors="none", alpha=0.78,
             label="H→H (n = 913)", zorder=2)
ax_b.scatter(x[valid & (evt==1)], y[valid & (evt==1)],
             s=10.5, c="#e52521", edgecolors="#9e1712", linewidths=0.35,
             label="H→C events (n = 83)", zorder=3)

# Centros por dente coloridos pela taxa observada.
for r in fig5_stats.itertuples():
    if np.isfinite(r.x) and np.isfinite(r.y):
        ax_b.scatter(
            [r.x],[r.y],s=13.0,c=[cmap(norm(r.event_fraction))],
            edgecolors="#4f5f6c",linewidths=0.28,zorder=4
        )

ax_b.legend(loc="upper center", fontsize=5.3, frameon=True,
            framealpha=0.96, borderpad=0.5, handletextpad=0.4)
ax_b.set_xticks([]); ax_b.set_yticks([])
ax_b.set_aspect("equal", adjustable="datalim")

sm = cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax_b, fraction=0.042, pad=0.028)
cbar.set_label("Observed H→C rate", fontsize=5.2)
cbar.ax.tick_params(labelsize=4.6, length=2)

# -------------------------- nota inferior --------------------------
ax_note.axis("off")
note_box = FancyBboxPatch(
    (0.005,0.06),0.99,0.88,transform=ax_note.transAxes,
    boxstyle="round,pad=0.004,rounding_size=0.018",
    facecolor="#fbfdff",edgecolor="#b8d1e6",linewidth=0.65
)
ax_note.add_patch(note_box)
ax_note.text(
    0.5,0.53,
    "Circle area ∝ eligible transitions (max n = 122). Fill = observed H→C fraction.\n"
    "Tooth labels: transitions / events. Heavy red outline: ≥1 observed event.\n"
    "All 83 events occurred in molars; zero observed events do not establish zero risk.",
    transform=ax_note.transAxes, ha="center", va="center",
    fontsize=4.45, color="#31465d"
)

# -------------------------- exportação --------------------------
fig5_dir = DIRS["figures"]
fig5_dir.mkdir(parents=True, exist_ok=True)

png_path = fig5_dir / "Figure_5_tooth_position_600dpi.png"
svg_path = fig5_dir / "Figure_5_tooth_position_vector.svg"
pdf_path = fig5_dir / "Figure_5_tooth_position_vector.pdf"
csv_path = fig5_dir / "Figure_5_data.csv"
audit_path = fig5_dir / "Figure_5_precheck.json"

# Não usar bbox_inches='tight': preserva exatamente 5.8 × 4.35 in => 3480 × 2610 px.
fig.savefig(png_path, dpi=FIG5_DPI, facecolor="white")
fig.savefig(svg_path, facecolor="white")
fig.savefig(pdf_path, facecolor="white")
plt.show()
plt.close(fig)

fig5_stats.to_csv(csv_path, index=False)

# Pre-check de resolução e integridade.
from PIL import Image
with Image.open(png_path) as img:
    px = tuple(img.size)
    dpi_meta = tuple(round(float(v),1) for v in img.info.get("dpi",(0,0)))

sha256 = hashlib.sha256(png_path.read_bytes()).hexdigest()
audit = {
    "file": str(png_path),
    "pixels": list(px),
    "dpi_metadata": list(dpi_meta),
    "display_width_in": FIG5_W_IN,
    "effective_ppi_at_5_8in": round(px[0]/FIG5_W_IN, 1),
    "expected_pixels": list(FIG5_EXPECTED_PX),
    "counts": {
        "rows": len(fig5_frame),
        "events": int(fig5_frame["progression_event"].sum()),
        "children": int(fig5_frame["child_id"].nunique()),
    },
    "sha256": sha256,
    "status": "PASS" if px == FIG5_EXPECTED_PX and px[0]/FIG5_W_IN >= 600 else "FAIL",
}
audit_path.write_text(json.dumps(audit, indent=2), encoding="utf-8")

assert audit["status"] == "PASS", audit
print(json.dumps(audit, indent=2))
print("PNG:", png_path)
print("SVG:", svg_path)
print("PDF:", pdf_path)
print("DATA:", csv_path)


## Submission reproduction checks

In [ ]:
import importlib.metadata, hashlib
from PIL import Image
assert PRIMARY_INTERNAL_VALIDATION["analysis_rows"] == 996
assert PRIMARY_INTERNAL_VALIDATION["analysis_events"] == 83
assert PRIMARY_INTERNAL_VALIDATION["analysis_children"] == 81
expected = {"clinical_minimal": (0.6864896607, 0.1664584675), "clinical_spatial": (0.7371501339, 0.1784215097)}
for name, (auc, ap) in expected.items():
    observed = PRIMARY_INTERNAL_VALIDATION["metrics"][name]["uncalibrated"]
    assert abs(observed["roc_auc"] - auc) < 0.002, (name, observed)
    assert abs(observed["pr_auc"] - ap) < 0.002, (name, observed)
assert abs(_marker_size(122)/_marker_size(12) - 122/12) < 1e-10
for t in ["T82", "T81", "T71", "T72"]:
    assert LOWER_MARKER[t][1] < LOWER_TOOTH[t][1]
with Image.open(png_path) as image:
    assert image.size == (3480,2610)
    assert abs(image.info["dpi"][0] - 600) < 1
packages = ["numpy","pandas","scipy","scikit-learn","matplotlib","openpyxl","requests","pillow"]
versions = {p:importlib.metadata.version(p) for p in packages}
(DIRS["reports"] / "runtime_versions.json").write_text(json.dumps(versions,indent=2))
print("PASS: cohort, primary metrics, proportional marker areas, lower tooth positions, native 600 dpi.")

for exported_png in [*OUT.glob("*.png"),png_path]:
    with Image.open(exported_png) as image:
        image.load()
        assert image.width / 5.8 >= 600
print("PASS: all scientific PNG files decode completely and meet effective resolution")
